# v34 GPU BASE advocate generation

Run this on a GPU account. It generates new advocates with the **base model** (no LoRA adapter), targeted at non-MC rows currently predicted `No` or high-risk YNU cases. It saves `v34_base_advocate_candidates.json` only; no selection happens here.

Why base model: v32.1 showed the LoRA adapter has anti-Yes bias. This notebook tests whether base Qwen can produce better candidate rationales.

In [ ]:
import json, re, math, os, sys, random, itertools, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

def find_file(names, required=True):
    if isinstance(names, str): names = [names]
    for d in CANDIDATE_DIRS:
        for name in names:
            p = d / name
            if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if d.exists():
            for root, dirs, files in os.walk(d):
                for name in names:
                    if name in files:
                        return Path(root)/name
    if required: raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f: return json.load(f), p

def save_json(obj, name):
    p = Path('/kaggle/working')/name
    try: p.parent.mkdir(parents=True, exist_ok=True)
    except Exception: pass
    with open(p, 'w', encoding='utf-8') as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    print('saved', p)
    return p

LABELS = ['A','B','C','D','Yes','No','Unknown']

def safe_pred(x):
    return x if x in LABELS else 'UNPARSEABLE'

def prf_for_label(rows, lab):
    tp=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')==lab)
    fp=sum(1 for r in rows if r.get('gold')!=lab and r.get('pred')==lab)
    fn=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')!=lab)
    prec=tp/(tp+fp) if tp+fp else 0.0
    rec=tp/(tp+fn) if tp+fn else 0.0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {'precision':prec,'recall':rec,'f1':f1,'support':tp+fn,'pred_count':tp+fp}

def metrics(rows):
    n=len(rows); acc=sum(1 for r in rows if r.get('gold')==r.get('pred'))/n
    per={lab: prf_for_label(rows, lab) for lab in LABELS}
    macro=sum(per[lab]['f1'] for lab in LABELS)/len(LABELS)
    weighted=sum(per[lab]['f1']*per[lab]['support'] for lab in LABELS)/sum(per[lab]['support'] for lab in LABELS)
    mc=[r for r in rows if r.get('is_mc')]
    ynu=[r for r in rows if not r.get('is_mc')]
    mc_labs=['A','B','C','D']; ynu_labs=['Yes','No','Unknown']
    mc_macro=sum(prf_for_label(mc, lab)['f1'] for lab in mc_labs)/4 if mc else 0
    ynu_macro=sum(prf_for_label(ynu, lab)['f1'] for lab in ynu_labs)/3 if ynu else 0
    return {'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro':mc_macro,'ynu_macro':ynu_macro,'per_label':per,
            'gold_count':dict(Counter(r.get('gold') for r in rows)), 'pred_count':dict(Counter(r.get('pred') for r in rows))}

def diffs(a,b):
    return [r1['idx'] for r1,r2 in zip(a,b) if r1.get('pred')!=r2.get('pred')]

def clone_rows(rows): return [dict(r) for r in rows]

def load_baselines():
    v27, p27 = load_json('v27_standard_preds.json', required=False)
    base, pb = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json','v32_2_standard_preds.json','v30_1_full_preds.json'], required=True)
    print('baseline path:', pb)
    base_m = metrics(base)
    print('baseline macro:', base_m['macro_f1'])
    return v27, p27, base, pb, base_m

BANKED = {14,71,109,125}
BASELINE_MACRO = 0.596858548901435

In [ ]:
v27, p27, base, pb, base_m = load_baselines()
# Target remaining No YNU excluding banked.
targets=[r for r in base if (not r.get('is_mc')) and r.get('pred')=='No' and r.get('idx') not in BANKED]
print('targets', len(targets), [r['idx'] for r in targets[:20]])

In [ ]:
# Model config. Edit BASE_MODEL if your Kaggle dataset path differs.
BASE_MODEL_CANDIDATES = [
    '/kaggle/input/qwen3-8b/transformers/default/1',
    '/kaggle/input/qwen-3/transformers/8b/1',
    '/kaggle/input/qwen3-8b',
    'Qwen/Qwen3-8B',
]

def pick_model_path():
    for p in BASE_MODEL_CANDIDATES:
        if p.startswith('/kaggle') and Path(p).exists(): return p
    return BASE_MODEL_CANDIDATES[-1]
BASE_MODEL = pick_model_path()
print('BASE_MODEL =', BASE_MODEL)

In [ ]:
# Load base model with safe fallback; no LoRA/PEFT.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model():
    modes=[]
    if torch.cuda.is_available():
        modes.append(('bf16_sharded', dict(device_map='auto', dtype=torch.bfloat16)))
        modes.append(('fp16_sharded', dict(device_map='auto', dtype=torch.float16)))
    modes.append(('cpu_float32', dict(device_map=None, dtype=torch.float32)))
    last=None
    for name,kw in modes:
        try:
            print('TRY', name, kw)
            tok=AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
            model=AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True, **kw)
            model.eval()
            return model,tok,name,None
        except Exception as e:
            print('FAILED', name, repr(e)[:500])
            last=repr(e)
    return None,None,None,last

model,tok,load_mode,load_error = load_model()
print('load_mode', load_mode, 'load_error', load_error)
assert model is not None, load_error

In [ ]:
# Strict JSON-ish prompt. We still parse with regex fallback.
def make_prompt(row, target):
    q=row.get('question','')
    ex=row.get('explanation','')[:2500]
    return f"""You are a strict logic verifier. Produce a rationale for the target answer only if it is actually entailed by the premises/context. Do not copy an inconsistent final answer.

QUESTION:
{q}

CURRENT MODEL EXPLANATION (may be wrong):
{ex}

TARGET_ANSWER: {target}

Return exactly this JSON object, no markdown:
{{"target_answer":"{target}","final_answer":"{target}","verdict":"VALID or INVALID","supporting_premises":[...],"reasoning":"..."}}
If the target answer is not logically supported, set verdict="INVALID" while keeping target_answer and final_answer fields.
"""

LABEL_RE=re.compile(r'"final_answer"\s*:\s*"(Yes|No|Unknown)"|Final Answer\s*[:\-]?\s*(Yes|No|Unknown)', re.I)
VERDICT_RE=re.compile(r'"verdict"\s*:\s*"(VALID|INVALID)"|VERDICT\s*[:\-]?\s*(VALID|INVALID)', re.I)

def parse_out(text):
    labs=[x for tup in LABEL_RE.findall(text) for x in tup if x]
    vers=[x for tup in VERDICT_RE.findall(text) for x in tup if x]
    return {'text':text, 'final_answer': labs[-1].title() if labs else None, 'verdict': vers[-1].upper() if vers else None,
            'cited': bool(re.search(r'\[[0-9,\s]+\]|supporting_premises', text, re.I))}

def generate_one(prompt, max_new_tokens=320):
    messages=[{"role":"user","content":prompt}]
    try:
        txt=tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        txt=tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=tok(txt, return_tensors='pt')
    if torch.cuda.is_available(): inputs={k:v.to(model.device) for k,v in inputs.items()}
    with torch.no_grad():
        out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None)
    gen=tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return gen

adv={'__meta__': {'version':'v34_base_advocate_generation','load_mode':load_mode,'load_error':load_error,'n_targets':len(targets)}}
for j,row in enumerate(targets):
    i=str(row['idx']); adv[i]={}
    print('GEN', j+1, '/', len(targets), 'idx', i)
    for target in ['Yes','No','Unknown']:
        gen=generate_one(make_prompt(row,target))
        adv[i][target]=parse_out(gen)
    if (j+1)%5==0: save_json(adv, 'v34_base_advocate_candidates.partial.json')
adv['__meta__']['n_generated']=len(targets)
save_json(adv, 'v34_base_advocate_candidates.json')